### 1.1. Import packages

Import the packages needed for this notebook. In addition to some standards (numpy, astropy), import lsst.afw packages for handling and plotting images, lsst packages for querying the table access protocol (TAP) service, and virtual observatory tools (pyvo) to enable the cutout service.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import io

from astropy import units as u
from astropy.wcs import WCS
from astropy.io import fits

import lsst.afw.display as afwDisplay
import lsst.afw.image as afwImage
from lsst.afw.image import ExposureF, ImageF
from lsst.afw.fits import MemFileManager

from lsst.rsp.utils import get_pyvo_auth
from lsst.rsp import get_tap_service

from pyvo.dal.adhoc import DatalinkResults, SodaQuery


### 1.2. Define parameters and functions

Create an instance of the TAP service, and assert that it exists.

In [ ]:
service = get_tap_service("tap")
assert service is not None

Set the backend for `afwDisplay` to `matplotlib`.

In [ ]:
afwDisplay.setDefaultBackend('matplotlib')

Define two wrapper functions to help later in the notebook. `get_cutout` will wrap the steps to call the cutout service and retrieve the image into a single function, and `make_subplot_grid` just sets up matplotlib to generate subplots for a specific number of cutouts.

In [ ]:
def get_cutout(dl_result, ra, dec, session, cutout_edge):
    sq = SodaQuery.from_resource(dl_result,
                                 dl_result.get_adhocservice_by_id("cutout-sync"),
                                 session=session)

    sq.circle = (ra * u.deg, dec * u.deg, cutout_edge * u.deg)


    # in theory these tests are for debugging
    try:
        sq.execute_stream().read()
    except Exception as e:
        print(e)
        if hasattr(e, 'response'):
            print(e.response.status_code)
            print(e.response.text)

    
    cutout_bytes = sq.execute_stream().read()
    sq.raise_if_error()
    mem = MemFileManager(len(cutout_bytes))
    mem.setData(cutout_bytes, len(cutout_bytes))
    hdul = fits.open(io.BytesIO(cutout_bytes))
    return ImageF(mem), hdul

In [ ]:
def make_subplot_grid(n_subplots, figsize_per_plot=(4, 3)):
    """
    Create an optimal grid of subplots for n_subplots.

    Returns
    -------
    fig : matplotlib.figure.Figure
    axes : list of matplotlib.axes.Axes
    """
    n_cols = math.ceil(math.sqrt(n_subplots))
    n_rows = math.ceil(n_subplots / n_cols)

    figsize = (figsize_per_plot[0] * n_cols,
               figsize_per_plot[1] * n_rows)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=figsize)
    
    axes = axes.flatten()

    for ax in axes[n_subplots:]:
        ax.remove()
    
    return fig, axes[:n_subplots]

## 2. Find visit images

### 2.1 Find the `ssObjectId` for the Solar System object of interest

Query the `MPCORB` table for the  object designation, orbital parameters, and `ssObjectId` for all DP1 Solar System objects.

In [ ]:
query = "SELECT mpc.mpcDesignation, "\
        "mpc.ssObjectId, "\
        "mpc.q, mpc.e, mpc.incl, "\
        "sso.discoverySubmissionDate, "\
        "sso.numObs "\
        "FROM dp1.MPCORB as mpc "\
        "INNER JOIN dp1.SSObject as sso "\
        "ON mpc.ssObjectId = sso.ssObjectId "\
        "ORDER BY mpc.mpcDesignation"

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

Fetch the job results and assign to an astropy `result` table. There are 431 Solar System objects in DP1.

In [ ]:
assert job.phase == 'COMPLETED'
result = job.fetch_result()
print(len(result))

Convert the `result` astropy table to a pandas dataframe.

In [ ]:
result_df = pd.DataFrame(result)

Print the results, sorting the table based on the parameters of interest. For example, find a new DP1 discovery with 10 observations.

In [ ]:
result_df.sort_values(by=["numObs"]).head(225).tail(25)

An object that satisfies our parameters of interest is 1991 SJ (ssObjectId = 20892032288436298), which is a bright Solar System object with 11 observations. See also 2024 WW106 (`ssObjectId` = 23133931615303986), which is a new Rubin DP1 discovery with 10 observations.


### 2.2 Find the visits for the `ssObjectId` of interest

Query the DiaSource catalog for the ra, dec, band, detector, and visit for each observation of the object of interest, inputting the ssObjectId for the object of interest (20892032288436298) into the query below.

In [ ]:
query = "SELECT mpc.mpcDesignation, "\
        "mpc.q, mpc.e, mpc.incl, "\
        "mpc.ssObjectId, "\
        "diasource.ssObjectId, "\
        "diasource.ra, diasource.dec, "\
        "diasource.band, diasource.detector, "\
        "diasource.visit, diasource.midpointMjdTai "\
        "FROM dp1.DiaSource as diasource "\
        "INNER JOIN dp1.MPCORB as mpc "\
        "ON diasource.ssObjectId = mpc.ssObjectId "\
        "WHERE diasource.ssObjectId = 20892032288436298 "\
        "ORDER BY diasource.midpointMjdTai"

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

Fetch the job results and assign to an astropy `result` table. There are 431 Solar System objects in DP1. There are 11 observations for 1991 SJ (ssObjectId = 20892032288436298) in DP1.


In [ ]:
assert job.phase == 'COMPLETED'
result = job.fetch_result()
print(len(result))

Convert the `result` to an astropy table.

In [ ]:
tab = result.to_table()

Option to print results.

In [ ]:
tab


## 3. Generate cutouts

## 3.1 Run a query for the first image

Start with the first visit to the SSObject. To generate a cutout, first retrieve the `access url` that points to the location of the image on the remote server.

In [ ]:
tab1 = tab[0] # first row of the table of visits
ra = tab1['ra']
dec = tab1['dec']

query2 = "SELECT dataproduct_type, dataproduct_subtype, calib_level, lsst_band, em_min, em_max, lsst_tract, lsst_patch, "\
         "lsst_filter, lsst_visit, lsst_detector, t_exptime, t_min, t_max, s_ra, s_dec, s_fov, obs_id, "\
         "obs_collection, o_ucd, facility_name, instrument_name, obs_title, s_region, access_url, access_format "\
         "FROM ivoa.ObsCore WHERE lsst_visit = "+str(tab1['visit'])+" AND lsst_detector = "+str(tab1['detector'])+" AND "\
         "obs_collection = 'LSST.DP1' AND calib_level = 2 AND dataproduct_type = 'image' AND "\
         "instrument_name = 'LSSTComCam' AND "\
         "dataproduct_subtype = 'lsst.visit_image' "\
         "ORDER by t_min ASC"

job = service.submit_job(query2)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
first_visit = job.fetch_result()
first_visit.to_table()


Retrieve the access url from the query (or datalink) which points to the image location on the remote server. Authorization to access the image is required and provided using get_pyvo_auth(). 

In [ ]:
datalink_url = first_visit["access_url"][0]

dl_result = DatalinkResults.from_result_url(datalink_url, session=get_pyvo_auth())
f"Datalink status: {dl_result.status}. Datalink service url: {datalink_url}"


Define the desired size of the cutout (we will indicate it is in degrees later). 

In [ ]:
cutout_edge = 0.002


Tell the remote server to make and return the cutout.

In [ ]:
sq = SodaQuery.from_resource(dl_result,
                             dl_result.get_adhocservice_by_id("cutout-sync-exposure"),
                             session=get_pyvo_auth())

sq.circle = (ra * u.deg, dec * u.deg, cutout_edge * u.deg)

cutout_bytes = sq.execute_stream().read()
sq.raise_if_error()

Receive the cutout "bytes" into memory, and convert it to an `ExposureF` image type.

In [ ]:
mem = MemFileManager(len(cutout_bytes))
mem.setData(cutout_bytes, len(cutout_bytes))
exposure = ExposureF(mem)

Display the science image extension of the `ExposureF` cutout using the Rubin/LSST image display package, `afwDisplay`.

In [ ]:
display = afwDisplay.Display()
display.scale('asinh', 'zscale')
display.image(exposure.image)
plt.show()

## 3.2 Generate cutout in a different format

Now, return the cutout in `ImageF` format instead of `ExposureF`. This is the same procedure, except one calls `get_adhocservice_by_id("cutout-sync")` instead of `get_adhocservice_by_id("cutout-sync-exposure")`, and the return format will be like a fits file. Use astropy and matplotlib tools to plot instead of LSST pipeline tools.

In [ ]:
sci, scihdul = get_cutout(dl_result, ra, dec, get_pyvo_auth(), cutout_edge)

sci_header = scihdul[1].header
scidata = scihdul[1].data

plt.subplot(1, 2, 1, projection=WCS(scihdul[1].header))
plt.imshow(scidata, origin='lower', vmin=np.nanpercentile(scidata, 1),
           vmax=np.nanpercentile(scidata, 99))


### 3.3 Generate cutouts from all visits to SSObject

Below, build the query for the full list of 10 images matching 10 coordinates / times (make sure sorted by time to match above). In the following cells, make use of the function `get_cutout` that was defined above to wrap over the commands demonstrated above to generate an ImageF type cutout.


In [ ]:
# Build a list of tuples: (visit, detector) that uniquely identify each visit image
points = []
for row in tab:
    points.append((row['visit'], row['detector']))

point_conditions = []

for visit, detector in points:
    cond = f"(lsst_visit = {visit} AND lsst_detector = {detector})"
    point_conditions.append(cond)
    
# Join all point conditions with OR
points_sql = " OR ".join(point_conditions)

query2 = f"""
SELECT dataproduct_type, dataproduct_subtype, calib_level, lsst_band, em_min, em_max,
       lsst_tract, lsst_patch, lsst_filter, lsst_visit, lsst_detector, t_exptime, t_min, t_max,
       s_ra, s_dec, s_fov, obs_id, obs_collection, o_ucd, facility_name, instrument_name,
       obs_title, s_region, access_url, access_format
FROM ivoa.ObsCore
WHERE obs_collection = 'LSST.DP1'
  AND calib_level = 2
  AND dataproduct_type = 'image'
  AND instrument_name = 'LSSTComCam'
  AND dataproduct_subtype = 'lsst.visit_image'
  AND ({points_sql})
ORDER BY t_min ASC
"""



Execute the query 

In [ ]:
job = service.submit_job(query2)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
if job.phase == 'ERROR':
    job.raise_if_error()
all_visits = job.fetch_result()
all_visits.to_table()


Iterate over the visits to generate a cutout for each (using the wrapper function `get_cutout` defined above), and plot the cutouts, in chronological order.

In [ ]:
fig, axes = make_subplot_grid(len(tab['ra']))

for i, ax in enumerate(axes):

    # retrieve data link that points to the image on remote server
    datalink_url = all_visits["access_url"][i]
    dl_result = DatalinkResults.from_result_url(datalink_url, session=get_pyvo_auth())
    f"Datalink status: {dl_result.status}. Datalink service url: {datalink_url}"

    # send datalink to the get_cutout wrapper that tells server to generate cutout
    sci, scihdul = get_cutout(dl_result, tab['ra'][i], tab['dec'][i], get_pyvo_auth(), cutout_edge)

    # get data from science image extensions
    sci_header = scihdul[1].header
    scidata = scihdul[1].data

    # plot cutouts using matplotlib
    ax.set_title(f"MJD {np.round(tab['midpointMjdTai'][i], 4)}")
    ax.imshow(scidata, origin='lower', vmin=np.nanpercentile(scidata, 1),
              vmax=np.nanpercentile(scidata, 99))

plt.show()